<a href="https://colab.research.google.com/github/shweta-shankar/MAS-AI-Research-Assistant/blob/gitSearch/gitSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pdfplumber groq

In [ ]:
from google.colab import userdata
from groq import Groq
import pdfplumber
import time

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

In [ ]:
from google.colab import files

print("Upload your gitingest txt file")
uploaded = files.upload()
gitingest_file = list(uploaded.keys())[0]

#print("Upload paper 1")
#uploaded = files.upload()
#pdf1 = list(uploaded.keys())[0]

#print("Upload paper 2")
#uploaded = files.upload()
#pdf2 = list(uploaded.keys())[0]

Upload your gitingest txt file


Saving goellab-digibone-8a5edab282632443.txt to goellab-digibone-8a5edab282632443 (1).txt


In [ ]:
def load_pdf(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                lines.append(f"================================================\n")
                lines.append(f"FILE: {pdf_path} (page {i+1})\n")
                lines.append(f"================================================\n")
                for line in text.splitlines(keepends=True):
                    lines.append(line)
    return lines

with open(gitingest_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

#lines += load_pdf(pdf1)
#lines += load_pdf(pdf2)

print(f"Total lines loaded: {len(lines)}")

Total lines loaded: 988


In [ ]:
def search(query, lines):
    results = []
    for i, line in enumerate(lines):
        if query.lower().replace(" ", "").replace("-", "") in line.lower().replace(" ", "").replace("-", ""):
            current_file = "unknown"
            for j in range(i, 0, -1):
                if lines[j].startswith("FILE:"):
                    current_file = lines[j].replace("FILE:", "").strip()
                    break
            start = max(0, i - 2)
            end = min(len(lines), i + 20)
            snippet = "".join(lines[start:end])
            results.append({
                "file": current_file,
                "line_number": i + 1,
                "snippet": snippet
            })
    return results


def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def extract_words(question):
    prompt = f"""Extract 3-5 keywords strictly from the question below.
Only use words that actually appear in the question.
Do NOT add related terms or your own knowledge.
Return ONLY a comma separated list, nothing else.

Question: {question}
"""
    return call_llm(prompt).split(",")


def filter_results(results, question):
    question_words = set(question.lower().replace("-","").replace(" ","").split())
    scored = []
    for r in results:
        snippet_words = set(r["snippet"].lower().replace("-","").replace(" ","").split())
        score = len(question_words & snippet_words)
        scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for score, r in scored[:8]]


def ask_llm(question, search_results):
    context = ""
    for r in search_results:
        context += f"\nFile: {r['file']} | Line: {r['line_number']}\n{r['snippet']}\n---"

    prompt = f"""You are a helpful assistant answering questions about a codebase and research papers.
Using ONLY the context below, write a clear and well structured answer.
- Start with the plain English definition or explanation first
- Then add implementation details and how it works in this specific project
- End every claim with a citation like [README.md:42]
- Do NOT start with code or file references
- If the context doesn't contain the answer, say so explicitly

Context:
{context}

Question: {question}
"""
    return call_llm(prompt)

In [ ]:
def run_query(question):
    start = time.time()

    keywords = extract_words(question)
    print("Keywords:", keywords)

    results = []
    for keyword in keywords:
        results += search(keyword, lines)

    # deduplicate
    seen = set()
    unique_results = []
    for r in results:
        key = (r["file"], r["line_number"])
        if key not in seen:
            seen.add(key)
            unique_results.append(r)

    print("Unique results:", len(unique_results))

    filtered = filter_results(unique_results, question)
    answer = ask_llm(question, filtered)

    end = time.time()
    print(f"\nAnswer:\n{answer}")
    print(f"\nTime taken: {(end-start)/60:.2f} mins")

# Ask your question here
run_query("How are short bone masks preprocessed to generate short-bone segments?")

Keywords: ['bone', ' masks', ' preprocessed', ' generate', ' short-bone']
Unique results: 94

Answer:
The short bone masks are preprocessed using a post-processing technique to generate the short-bone segments. This involves subtracting the carpal and wrist crops from the full-hand image to obtain the shortbones crop of the image [README.md:28]. The specifics of the post-processing technique are implemented in the `segment_crops.py` file, which is used to generate the carpal and wrist crops of Indian X-ray images after appropriate mask post-processing [README.md:28].

Time taken: 0.02 mins


In [ ]:
run_query("What is the difference between the scoring system in this method and the standard GP method used by a doctor? What can I expect to be the same and what to be different ")

Keywords: ['difference', ' scoring', ' system', ' method', ' doctor']
Unique results: 8

Answer:
The Segmental Greulich-Pyle (SGP) method used in this project differs from the standard Greulich-Pyle (GP) method in how it scores bone age. In the standard GP method, a doctor compares a child's full-hand X-ray to reference radiographs to assign a single bone age [README.md:17]. However, this method can be time-consuming and prone to inter- and intra-observer variability [README.md:15]. 

In contrast, the SGP method divides the full hand into three meaningful anatomical segments based on their maturational variation, namely, Shortbones (Metacarpal and Phalanges), Carpals, and Wrist (Radius and Ulna) [README.md:20]. The SGP method then determines the segmental bone ages for each of these segments and combines them to report a more appropriate bone age of the child [README.md:20]. This is different from the standard GP method, which only provides a single bone age for the full hand [README.m

In [11]:
run_query("can this system be used to estimate the bone age of mummies?")

Keywords: ['system', ' bone', ' age', ' mummies']
Unique results: 183

Answer:
Estimating the bone age of mummies is a process that typically involves analyzing the skeletal remains to determine the individual's age at the time of death. However, the system described in the context is designed to assess bone age in children using X-ray images of their hands, specifically for monitoring growth and development in pediatric patients [README.md:1]. 

The system uses a Segmental Greulich-Pyle (SGP) method, which divides the full hand into three meaningful anatomical segments based on their maturational variation, namely, Shortbones (Metacarpal and Phalanges), Carpals and Wrist (Radius and Ulna) [README.md:5]. It is unlikely that this system could be directly applied to estimate the bone age of mummies, as it requires X-ray images of the hand, which may not be feasible or relevant for mummified remains [README.md:10].

Furthermore, the system is trained on datasets of X-ray images from livin

In [12]:
run_query("what is the basketball match score")

Keywords: ['basketball', ' match', ' score']
Unique results: 0

Answer:
The basketball match score refers to the total number of points earned by each team during a game, with the team having the higher score at the end of the game being declared the winner [README.md:42]. 

In the context of this project, the implementation details of the basketball match score are not available, as there is no information provided about how the score is calculated or tracked [README.md:42]. 

Unfortunately, the context does not contain the current or final score of a specific basketball match [README.md:42].

Time taken: 0.01 mins


In [13]:
run_query("How were the segmental GP labels assigned to each image?")

Keywords: ['segmental', ' GP', ' labels', ' image']
Unique results: 105

Answer:
In the context of bone age assessment, segmental GP labels refer to the assignment of bone age labels to specific segments of the hand, such as short bones, carpals, and wrist. The segmental GP labels were assigned to each image by using a novel method called Segmental Greulich-Pyle (SGP), which divides the full hand into three meaningful anatomical segments based on their maturational variation [README.md:20]. 

The SGP method uses a combination of segmentation models and bone age prediction models to assign segmental bone ages. The segmentation models, which have a Unet backbone, are used to segment the carpals and wrist regions of the X-ray images [README.md:40]. The bone age prediction models, which have a densenet161 backbone, are then used to predict the bone age for each segment [README.md:49]. 

The segmental predictions are equally weighted and rounded off to their nearest GP class to obtain the f